# 30 秒手势 + 切分点动态 render

对应参考 notebook 4.1：左侧 Manus 25 关节 skeleton 动画，右侧从上到下是 pose speed（带阈值线与切分背景）、EMG RMS 包络、当前 ergonomics 通道柱图，所有曲线共用一根红色时间游标。

完全复用项目内的 `emg_label` 模块（[pose_segmentation](emg_label/pose_segmentation.py)、[segmentation.emg_envelope](emg_label/segmentation.py#L7)、[io_utils.load_npz / load_skeleton](emg_label/io_utils.py#L148)、[skeleton.draw_skeleton](emg_label/skeleton.py#L47)），不依赖 `reference/gesture_velocity_segmentation.py`。

In [1]:
%matplotlib inline
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML, display
from matplotlib.animation import FuncAnimation

# 让 emg_label 可导入（notebook 在项目根目录时自动生效）
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from emg_label import io_utils, pose_segmentation, segmentation, skeleton

plt.rcParams["axes.grid"] = True

## 1. 选择一条 raw npz

两种输入：
- 直接给 `npz_path`（并设置 `explicit_hand`），把 `overview_png` 设为 `None`；
- 给 `overview_png`（比如 `out_pose/overview/zxc-0607__20260427-left__20260427_105740.png`），用 `stem_to_npz` 反推回 `{NPZ_ROOT}/{subject}/{session}/{filename}.npz`，同时从 `session` 拿到 `hand`。

`fs` 取项目默认的 EMG 采样率，pose 和 skeleton 都会被 `load_npz` / `load_skeleton` 对齐到 EMG 时间轴上，所以 `t = np.arange(n) / fs` 是统一的时间轴。

In [2]:
# 输入二选一：
#   (1) 直接给 npz_path（并自行指定 hand）
#   (2) 给 overview_png，自动反推出 npz_path 和 hand
# raw npz 路径布局：{NPZ_ROOT}/{subject}/{session}/{filename}.npz
# overview png 名约定：{subject}__{session}__{filename}.png（io_utils.parse_file_info 合成）
NPZ_ROOT = Path("/mnt/pose_data/emg2pose/data")

overview_png = Path(
    # "/data/cl_data/action-clustering/out_pose/overview/zxc-0607__20260427-left__20260427_105740.png"
    # "/data/cl_data/action-clustering/out_pose/overview/pgy-0226__20260426-left-1__20260426_152119.png"
    "/data/cl_data/action-clustering/out_pose2/shards/ax-0819__20260428-left-3__20260428_140343"
)
npz_path = None        # 想直接指定 npz 路径就填这里、并把 overview_png 设为 None
explicit_hand = None   # 不为 None 时覆盖从 session 反推出来的 hand


def stem_to_npz(stem, root=NPZ_ROOT):
    """`{subject}__{session}__{filename}` -> ({root}/subject/session/filename.npz, hand)."""
    parts = stem.split("__")
    if len(parts) < 3:
        raise ValueError(f"stem 至少要 subject__session__filename 三段: {stem!r}")
    subject, session, filename = parts[0], parts[1], "__".join(parts[2:])
    seg = session.split("-")
    hand = seg[1].lower() if len(seg) >= 2 and seg[1].lower() in ("left", "right", "both") else None
    return root / subject / session / f"{filename}.npz", hand


if overview_png is not None:
    npz_path, hand = stem_to_npz(Path(overview_png).stem)
else:
    assert npz_path is not None, "overview_png 与 npz_path 至少给一个"
    hand = "left"
if explicit_hand is not None:
    hand = explicit_hand

fs = 2000               # EMG 采样率
max_time_s = 60         # render 的时间窗口长度（真实秒数）

assert npz_path.exists(), f"找不到 npz: {npz_path}"
print(f"file: {npz_path}\nhand: {hand}, fs: {fs}")

file: /mnt/pose_data/emg2pose/data/ax-0819/20260428-left-3/20260428_140343.npz
hand: left, fs: 2000


## 2. 加载数据 + 跑切分

- `io_utils.load_npz` 返回对齐到 EMG 轴的 `(emg, ja)`。
- `io_utils.load_skeleton` 返回 `(T, 25, 3)` 的 Manus skeleton；若 npz 不含 `manus_*_skeleton` 会返回 `None`。
- `pose_segmentation.pose_speed / robust_threshold / static_motion_intervals` 复刻参考 notebook 里的速度切分。
- `segmentation.emg_envelope` 是 baseline-centered RMS（公式和 4.1 用的 `robust_emg_envelope` 一致）。

In [3]:
emg, ja = io_utils.load_npz(str(npz_path), hand=hand)
skel = io_utils.load_skeleton(str(npz_path), hand=hand)
n = len(emg)
t_axis = np.arange(n) / fs

speed = pose_segmentation.pose_speed(ja, fs, smooth_ms=250.0)
thr = pose_segmentation.robust_threshold(speed, percentile=35.0, mad_scale=1.5)
static_iv, motion_iv = pose_segmentation.static_motion_intervals(
    speed, fs, thr, min_static_s=0.35, min_motion_s=0.20, merge_gap_s=0.20,
)
env = segmentation.emg_envelope(emg, fs=fs, smooth_ms=150.0)

print(f"emg: {emg.shape}, ja: {ja.shape}, skeleton: {None if skel is None else skel.shape}")
print(f"duration: {n / fs:.1f}s, threshold: {thr:.4f}")
print(f"static segs: {len(static_iv)}, motion segs: {len(motion_iv)}")
if skel is None:
    print("⚠️  这条 npz 没有 manus_*_skeleton，下面会跳过 3D 手部子图。")

emg: (107200, 16), ja: (107200, 20), skeleton: (107200, 25, 3)
duration: 53.6s, threshold: 1002.3352
static segs: 16, motion segs: 15


## 3. 渲染 30 秒动态图（1:1 真实速度）

播放时长 = `max_frames / fps`，数据窗口 = `max_time_s`。这里 `max_frames=600 / fps=20`，对应 30s 播放、刚好覆盖 30s 数据（不加速）。3D skeleton 每帧都要重绘，600 帧的 jshtml 在我们这条数据上约 150–170 MB，所以 `animation.embed_limit` 提到 256 MB；如果浏览器吃不消，可以把 `fps` 调到 15 或 10，相应把 `max_frames` 改成 `30*fps`，仍是真实速度。

In [ ]:
plt.rcParams["animation.embed_limit"] = 256  # MB（30s × 20fps × 3D 重绘 ≈ 150–170 MB）


def _robust_upper(x, percentile=98.5, min_upper=None):
    x = np.asarray(x, dtype=np.float64)
    finite = x[np.isfinite(x)]
    if finite.size == 0:
        return 1.0
    upper = float(np.nanpercentile(finite, percentile))
    if min_upper is not None:
        upper = max(upper, float(min_upper))
    return upper if np.isfinite(upper) and upper > 0 else 1.0


def render_60s(emg, ja, env, speed, thr, static_iv, motion_iv, skel,
               fs, t_axis, max_time_s=60, fps=20, max_frames=240,
               speed_p=98.5, emg_p=98.5):
    end_t = float(min(max_time_s, t_axis[-1]))
    mask = t_axis <= end_t
    t_view = t_axis[mask]
    speed_v, env_v, ja_v = speed[mask], env[mask], ja[mask]

    # 采样时间游标的帧位置（按时间均匀，再映射回样本下标）
    n_frames = min(max_frames, len(t_view))
    frame_t = np.linspace(float(t_view[0]), float(t_view[-1]), n_frames)
    frame_idx = np.clip(np.round(frame_t * fs).astype(int), 0, len(t_axis) - 1)

    speed_upper = _robust_upper(speed_v, speed_p, min_upper=thr)
    emg_upper = _robust_upper(env_v, emg_p)
    ja_low = float(np.nanpercentile(ja_v, 1))
    ja_high = float(np.nanpercentile(ja_v, 99))
    if not np.isfinite(ja_low) or not np.isfinite(ja_high) or ja_low == ja_high:
        ja_low, ja_high = float(np.nanmin(ja_v)), float(np.nanmax(ja_v))
    ja_margin = max((ja_high - ja_low) * 0.12, 1e-3)

    have_skel = skel is not None
    skel_view = None
    skel_limits = None
    if have_skel:
        skel_view = skeleton.normalize_skeleton(skel) * 1000.0  # m -> mm
        skel_limits = skeleton.axis_limits(skel_view[mask])

    fig = plt.figure(figsize=(17, 9))
    if have_skel:
        gs = fig.add_gridspec(3, 2, width_ratios=[1.08, 1.55],
                              height_ratios=[1.0, 0.9, 1.05])
        ax_hand = fig.add_subplot(gs[:, 0], projection="3d")
        ax_speed = fig.add_subplot(gs[0, 1])
        ax_emg = fig.add_subplot(gs[1, 1], sharex=ax_speed)
        ax_pose = fig.add_subplot(gs[2, 1])
    else:
        gs = fig.add_gridspec(3, 1, height_ratios=[1.0, 0.9, 1.05])
        ax_hand = None
        ax_speed = fig.add_subplot(gs[0, 0])
        ax_emg = fig.add_subplot(gs[1, 0], sharex=ax_speed)
        ax_pose = fig.add_subplot(gs[2, 0])

    timeline_axes = [ax_speed, ax_emg]

    # 切分段：static_hold 绿色、transition_motion 橙色
    seg_specs = (
        [(s, e, "static_hold", "tab:green", 0.20) for (s, e) in static_iv]
        + [(s, e, "transition_motion", "tab:orange", 0.12) for (s, e) in motion_iv]
    )
    for s_idx, e_idx, _name, color, alpha in seg_specs:
        s_t = max(0.0, s_idx / fs)
        e_t = min(end_t, e_idx / fs)
        if e_t <= s_t:
            continue
        for ax in timeline_axes:
            ax.axvspan(s_t, e_t, color=color, alpha=alpha, lw=0)
            ax.axvline(s_t, color=color, lw=0.6, alpha=0.55)
            ax.axvline(e_t, color=color, lw=0.6, alpha=0.55)

    ax_speed.plot(t_view, speed_v, lw=0.9, color="black", label="pose speed")
    ax_speed.axhline(thr, color="crimson", ls="--", lw=1.1, label="threshold")
    ax_speed.set_ylim(0, speed_upper * 1.12)
    ax_speed.set_xlim(0, end_t)
    ax_speed.set_ylabel("pose speed")
    ax_speed.legend(loc="upper right", fontsize=8)

    ax_emg.plot(t_view, env_v, lw=0.85, color="tab:blue", label="EMG RMS envelope")
    ax_emg.set_ylim(0, emg_upper * 1.12)
    ax_emg.set_xlim(0, end_t)
    ax_emg.set_ylabel("EMG RMS")
    ax_emg.set_xlabel("time (s)")
    ax_emg.legend(loc="upper right", fontsize=8)

    ch_x = np.arange(ja.shape[1])
    pose_line, = ax_pose.plot(ch_x, ja[frame_idx[0]], marker="o", lw=1.4, color="tab:blue")
    ax_pose.set_xlim(-0.5, ja.shape[1] - 0.5)
    ax_pose.set_ylim(ja_low - ja_margin, ja_high + ja_margin)
    ax_pose.set_xlabel("ergonomics channel")
    ax_pose.set_ylabel("current pose")

    cursors = [ax.axvline(float(frame_t[0]), color="crimson", lw=1.8) for ax in timeline_axes]
    title = fig.suptitle("")

    def update(i):
        idx = int(frame_idx[i])
        cur_t = float(frame_t[i])
        if have_skel:
            ax_hand.clear()
            skeleton.draw_skeleton(ax_hand, skel_view[idx])
            (xlo, xhi), (ylo, yhi), (zlo, zhi) = skel_limits
            ax_hand.set_xlim(xlo, xhi)
            ax_hand.set_ylim(ylo, yhi)
            ax_hand.set_zlim(zlo, zhi)
            ax_hand.set_xticks([]); ax_hand.set_yticks([]); ax_hand.set_zticks([])
            ax_hand.set_title(f"manus skeleton | t={cur_t:.2f}s | frame {idx}")
        pose_line.set_ydata(ja[idx])
        for c in cursors:
            c.set_xdata([cur_t, cur_t])
        title.set_text(f"{npz_path.name} | t={cur_t:.2f}s / {end_t:.0f}s")
        return [pose_line, *cursors, title]

    anim = FuncAnimation(fig, update, frames=n_frames, interval=1000 / fps, blit=False)
    plt.close(fig)
    return anim

fps=10
duration=max_time_s
anim = render_60s(
    emg, ja, env, speed, thr, static_iv, motion_iv, skel,
    fs=fs, t_axis=t_axis, max_time_s=max_time_s, fps=fps, max_frames=duration*fps,
)
HTML(anim.to_jshtml())